# RAG-Based Text-to-SQL Generator
**Concept:** Upload a .docx with existing SQL queries → RAG retrieves the best match → LLM modifies it for a new client

---

## STEP 1: Install Dependencies

In [1]:
# Install all required libraries
!pip install python-docx openai faiss-cpu numpy -q

# python-docx  → read .docx files
# openai       → embeddings + GPT
# faiss-cpu    → vector store (similarity search)
# numpy        → array/vector math

print('All libraries installed!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 23.2 MB/s eta 0:00:00
All libraries installed!


## STEP 2: Set OpenAI API Key

In [2]:
# STEP 2: Install Gemini
!pip install google-generativeai -q

import google.generativeai as genai

# Free API key → https://aistudio.google.com/app/apikey
# Sign in with Google → "Get API Key" → Copy it

GEMINI_API_KEY = "sample"
genai.configure(api_key=GEMINI_API_KEY)

print("Gemini ready!")

Gemini ready!


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


## STEP 3: Upload & Parse the .docx File

In [3]:
from google.colab import files
from docx import Document
import re

# Upload your .docx file
uploaded = files.upload()
filename = list(uploaded.keys())[0]   # get the uploaded filename

# Load the Word document
doc = Document(filename)

# ----- Parse into structured documents -----
# We look for headings like: 'Client: Kate | Query: SKU Status'
# Then grab the SQL that follows

raw_documents = []   # will store list of dicts
current_meta = {}
current_sql_lines = []

for para in doc.paragraphs:
    text = para.text.strip()
    if not text:
        continue

    # Detect heading lines like "Client: Kate | Query: SKU Status"
    if 'Client:' in text and 'Query:' in text:
        # Save previous document if exists
        if current_meta and current_sql_lines:
            raw_documents.append({
                'client': current_meta['client'],
                'query_name': current_meta['query_name'],
                'sql': '\n'.join(current_sql_lines).strip()
            })

        # Parse new heading
        # Format: "Client: Kate | Query: SKU Status"
        parts = text.split('|')
        current_meta = {
            'client': parts[0].replace('Client:', '').strip(),
            'query_name': parts[1].replace('Query:', '').strip()
        }
        current_sql_lines = []   # reset SQL buffer
    else:
        # Everything else is SQL content
        current_sql_lines.append(text)

# Don't forget the last block
if current_meta and current_sql_lines:
    raw_documents.append({
        'client': current_meta['client'],
        'query_name': current_meta['query_name'],
        'sql': '\n'.join(current_sql_lines).strip()
    })

print(f'Parsed {len(raw_documents)} queries from the document:\n')
for d in raw_documents:
    print(f"  Client: {d['client']} | Query: {d['query_name']}")

Saving sample_queries.docx to sample_queries (1).docx
Parsed 4 queries from the document:

  Client: Kate | Query: SKU Status
  Client: Kate | Query: Revenue Summary
  Client: Nike | Query: SKU Status
  Client: Nike | Query: Top Sellers


## STEP 4: Create LangChain-style Documents (page_content + metadata)

In [4]:
# Mimic LangChain's Document structure manually (no LangChain needed!)
# Each document = { page_content: SQL, metadata: { client, query_name } }

documents = []

for q in raw_documents:
    doc_entry = {
        # page_content = what gets embedded (semantic search runs on this)
        'page_content': f"Query: {q['query_name']}\n{q['sql']}",

        # metadata = used for filtering (exact match, not semantic)
        'metadata': {
            'client': q['client'],
            'query_name': q['query_name']
        }
    }
    documents.append(doc_entry)

print('Documents created:')
for d in documents:
    print(f"  {d['metadata']}")

Documents created:
  {'client': 'Kate', 'query_name': 'SKU Status'}
  {'client': 'Kate', 'query_name': 'Revenue Summary'}
  {'client': 'Nike', 'query_name': 'SKU Status'}
  {'client': 'Nike', 'query_name': 'Top Sellers'}


In [7]:
# Check which embedding models you have access to
for m in genai.list_models():
    if 'embedContent' in m.supported_generation_methods:
        print(m.name)

models/gemini-embedding-001
models/gemini-embedding-2-preview
models/gemini-embedding-2


## STEP 5: Generate Embeddings (page_content → vectors)

In [8]:
import numpy as np
import google.generativeai as genai

def get_embedding(text):
    result = genai.embed_content(
        model="models/gemini-embedding-001",  # ← use this
        content=text
    )
    return result['embedding']

# Embed each document's page_content
print('Generating embeddings...')
for doc_entry in documents:
    doc_entry['embedding'] = get_embedding(doc_entry['page_content'])
    print(f"  Embedded: {doc_entry['metadata']['client']} - {doc_entry['metadata']['query_name']}")

# Embedding shape check
dim = len(documents[0]['embedding'])
print(f'\nEmbedding dimensions: {dim}')   # will be 768 (Gemini uses 768, OpenAI uses 1536)

Generating embeddings...
  Embedded: Kate - SKU Status
  Embedded: Kate - Revenue Summary
  Embedded: Nike - SKU Status
  Embedded: Nike - Top Sellers

Embedding dimensions: 3072


## STEP 6: Store in FAISS Vector DB

In [9]:
import faiss

# Stack all embeddings into a 2D numpy array: shape = (num_docs, 1536)
vectors = np.array(
    [d['embedding'] for d in documents],
    dtype='float32'
)

# Create FAISS index
# IndexFlatL2 = exact search using L2 (Euclidean) distance
dim = vectors.shape[1]           # number of embedding dimensions
index = faiss.IndexFlatL2(dim)

# Add all document vectors to the index
index.add(vectors)

print(f'FAISS index ready! Total vectors stored: {index.ntotal}')

FAISS index ready! Total vectors stored: 4


## STEP 7: Retrieval — Find Best Matching Query

In [10]:
def retrieve(user_prompt, filter_query_name=None, top_k=3):
    """
    Retrieve most relevant documents for a user prompt.

    Steps:
    1. (Optional) Metadata filter → narrow candidates
    2. Embed user query
    3. Similarity search in FAISS
    4. Return top matches
    """

    # --- Step 1: Metadata filtering (optional) ---
    # Filter documents by query_name before semantic search
    if filter_query_name:
        candidates = [
            (i, d) for i, d in enumerate(documents)
            if d['metadata']['query_name'].lower() == filter_query_name.lower()
        ]
    else:
        candidates = list(enumerate(documents))

    if not candidates:
        print('No documents matched the filter!')
        return []

    # --- Step 2: Embed user query ---
    query_vector = np.array(
        [get_embedding(user_prompt)],
        dtype='float32'
    )  # shape: (1, 1536)

    # --- Step 3: Build sub-index from filtered candidates ---
    # (For simplicity, we search only among filtered docs)
    filtered_vectors = np.array(
        [documents[i]['embedding'] for i, _ in candidates],
        dtype='float32'
    )
    sub_index = faiss.IndexFlatL2(filtered_vectors.shape[1])
    sub_index.add(filtered_vectors)

    # --- Step 4: Similarity search ---
    k = min(top_k, len(candidates))   # can't retrieve more than we have
    distances, indices = sub_index.search(query_vector, k)

    # Map back to original documents
    results = []
    for rank, idx in enumerate(indices[0]):
        orig_idx, orig_doc = candidates[idx]
        results.append({
            'rank': rank + 1,
            'distance': distances[0][rank],
            'document': orig_doc
        })

    return results

print('Retrieval function ready!')

Retrieval function ready!


## STEP 8: Generate New SQL for Coach (RAG + LLM)

In [15]:
# ---- USER INPUT ----
new_client = 'Coach'
reference_client = 'Kate'
query_name_filter = 'SKU Status'

user_prompt = f"""
Generate SKU status SQL for client {new_client}.
Use {reference_client}'s SKU status logic as reference,
but change the OLD tag to: not sold for last 5 years.
Keep everything else the same.
"""

# ---- RETRIEVAL ----
results = retrieve(
    user_prompt=user_prompt,
    filter_query_name=query_name_filter,
    top_k=1
)

# Show what was retrieved
retrieved_doc = results[0]['document']
print('=== Retrieved Document ===')
print(f"Client: {retrieved_doc['metadata']['client']}")
print(f"Query : {retrieved_doc['metadata']['query_name']}")
print(f"SQL   :\n{retrieved_doc['page_content']}")

# ---- BUILD PROMPT FOR LLM ----
prompt = f"""You are a SQL expert.

Here is an existing SQL query used by client {reference_client}:

{retrieved_doc['page_content']}

Now generate a modified version for client {new_client}.

Changes required:
- Change the 'OLD' tag condition to: not sold for last 5 years (i.e., last_sale_date < CURRENT_DATE - INTERVAL '5 years')
- Replace the tag value 'OLD' with 'NOT_SOLD_5YRS'
- Keep all other logic the same

Return ONLY the final SQL. No explanation."""



model = genai.GenerativeModel("models/gemini-2.5-flash")
response = model.generate_content(prompt)
generated_sql = response.text

print('\n=== Generated SQL for Coach ===')
print(generated_sql)

=== Retrieved Document ===
Client: Kate
Query : SKU Status
SQL   :
Query: SKU Status
SELECT sku,
       CASE 
           WHEN last_sale_date < CURRENT_DATE - INTERVAL '365 days'
           THEN 'OLD'
           ELSE 'ACTIVE'
       END AS status
FROM sales;

=== Generated SQL for Coach ===
```sql
SELECT sku,
       CASE 
           WHEN last_sale_date < CURRENT_DATE - INTERVAL '5 years'
           THEN 'NOT_SOLD_5YRS'
           ELSE 'ACTIVE'
       END AS status
FROM sales;
```


In [13]:
for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gem

In [19]:
!pip install groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 3.6 MB/s eta 0:00:00


## STEP 9: Wrap as a Reusable Function
Now let's put everything together into one clean function.

In [20]:
from groq import Groq

groq_client = Groq(api_key="sample")  # https://console.groq.com
print("Groq ready!")

Groq ready!


In [21]:
def generate_sql_for_new_client(
    new_client: str,
    reference_client: str,
    query_type: str,
    modifications: str
):
    # Step 1: User prompt for retrieval
    user_prompt = f"""
    Generate {query_type} SQL for {new_client}.
    Based on {reference_client} logic. Changes: {modifications}
    """

    # Step 2: Retrieve
    results = retrieve(
        user_prompt=user_prompt,
        filter_query_name=query_type,
        top_k=1
    )

    if not results:
        return 'No matching query found in the document.'

    retrieved_doc = results[0]['document']

    # Step 3: Build LLM prompt
    prompt = f"""You are a SQL expert.

Reference SQL ({reference_client} - {query_type}):
{retrieved_doc['page_content']}

Generate a modified SQL for client: {new_client}
Modifications: {modifications}
Keep rest same.
Return ONLY SQL."""

    # Step 4: Call Groq instead of OpenAI
    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content


# ---- TEST IT ----
result = generate_sql_for_new_client(
    new_client='Coach',
    reference_client='Kate',
    query_type='SKU Status',
    modifications="Change OLD tag to NOT_SOLD_5YRS for items not sold in last 5 years"
)

print('Generated SQL:')
print(result)

Generated SQL:
```sql
SELECT sku,
       CASE 
           WHEN last_sale_date < CURRENT_DATE - INTERVAL '5 years'
           THEN 'NOT_SOLD_5YRS'
           ELSE 'ACTIVE'
       END AS status
FROM sales;
```


---
## Summary: What happened?

```
.docx file
    ↓  Parse with python-docx
Documents (page_content + metadata)
    ↓  OpenAI Embeddings
Vectors stored in FAISS
    ↓  User asks for new client SQL
Metadata filter (query_name = SKU Status)
    +  Semantic search (embedding similarity)
    ↓
Best matching SQL retrieved
    ↓  Build prompt with retrieved SQL
GPT-4o-mini modifies it
    ↓
New SQL for Coach ✅
```

**Key concepts for interview:**
- `page_content` → embedded (semantic search)
- `metadata` → filtered (exact match)
- FAISS → fast vector similarity search
- RAG = retrieve first, then generate (not hallucinate)
